# Xarray-Spatial Sky View: Sky-view factor computation

Sky-view factor (SVF) measures the fraction of the sky hemisphere visible from each cell in a DEM. Values range from 0 (fully obstructed, like a deep canyon) to 1 (flat open terrain with no horizon obstruction). SVF is used in LiDAR archaeology to reveal subtle terrain features, urban heat island studies, and solar energy modeling as a proxy for diffuse sky irradiance.

### What you'll build

1. Generate a synthetic DEM with ridges, valleys, and a central peak
2. Compute SVF with default parameters
3. Compare the effect of search radius on SVF
4. Compare the effect of azimuth direction count on smoothness
5. Side-by-side comparison of SVF vs. hillshade

![Sky view factor preview](images/sky_view_factor_preview.png)

**Jump to a section:**
[Default SVF](#Default-SVF) | [Search radius](#Search-radius) | [Direction count](#Direction-count) | [SVF vs. hillshade](#SVF-vs.-hillshade)

Standard imports plus `sky_view_factor` and `hillshade` from xrspatial.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from xrspatial import sky_view_factor, hillshade

## Synthetic terrain

Sinusoidal ridges plus a central Gaussian peak give a surface with valleys, ridges, and a flat plateau to show how SVF responds to different landforms.

In [ ]:
rows, cols = 200, 200
Y, X = np.meshgrid(np.arange(rows, dtype=float), np.arange(cols, dtype=float), indexing='ij')

terrain = (
    40 * np.sin(X / 15) * np.cos(Y / 20)
    + 80 * np.exp(-((Y - 100)**2 + (X - 100)**2) / (2 * 30**2))
    + 20 * np.sin(Y / 10)
    + 200
)

dem = xr.DataArray(
    terrain,
    dims=['y', 'x'],
    coords={'y': np.arange(rows, dtype=float), 'x': np.arange(cols, dtype=float)},
    attrs={'res': (1.0, 1.0)},
)

fig, ax = plt.subplots(figsize=(10, 7.5))
dem.plot.imshow(ax=ax, cmap='terrain', add_colorbar=True,
                cbar_kwargs={'label': 'Elevation'})
ax.set_title('Synthetic DEM')
ax.set_axis_off()
plt.tight_layout()

## Default SVF

The default computation uses `max_radius=10` cells and `n_directions=16` azimuth rays. Dark areas (low SVF) are in valleys where the horizon is obstructed; bright areas (high SVF) are on ridges and the flat plateau.

In [ ]:
svf = sky_view_factor(dem, max_radius=10, n_directions=16)

fig, ax = plt.subplots(figsize=(10, 7.5))
svf.plot.imshow(ax=ax, cmap='gray', vmin=0, vmax=1, add_colorbar=True,
                cbar_kwargs={'label': 'Sky-View Factor'})
ax.set_title('SVF (max_radius=10, n_directions=16)')
ax.set_axis_off()
plt.tight_layout()

## Search radius

A larger `max_radius` captures more distant obstructions. This matters in terrain with broad features like wide valleys. Increasing the radius also increases computation time quadratically.

In [ ]:
radii = [5, 15, 30]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, r in zip(axes, radii):
    result = sky_view_factor(dem, max_radius=r, n_directions=16)
    result.plot.imshow(ax=ax, cmap='gray', vmin=0, vmax=1, add_colorbar=False)
    ax.set_title(f'max_radius={r}', fontsize=12)
    ax.set_axis_off()

plt.suptitle('SVF at different search radii', fontsize=14, y=1.02)
plt.tight_layout()

## Direction count

More azimuth directions give a smoother result at the cost of longer computation. With 4 directions the result has visible angular artifacts. 16 directions is usually a good balance between quality and speed.

In [ ]:
directions = [4, 16, 64]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, nd in zip(axes, directions):
    result = sky_view_factor(dem, max_radius=15, n_directions=nd)
    result.plot.imshow(ax=ax, cmap='gray', vmin=0, vmax=1, add_colorbar=False)
    ax.set_title(f'n_directions={nd}', fontsize=12)
    ax.set_axis_off()

plt.suptitle('SVF at different direction counts', fontsize=14, y=1.02)
plt.tight_layout()

## SVF vs. hillshade

Hillshade depends on a specific sun angle, which creates directional bias: features aligned with the light source can disappear. SVF provides omnidirectional illumination information, making features visible regardless of orientation. This is why SVF is preferred for archaeological feature detection.

In [ ]:
hs = hillshade(dem, azimuth=315, angle_altitude=45)
svf_final = sky_view_factor(dem, max_radius=15, n_directions=16)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

hs.plot.imshow(ax=axes[0], cmap='gray', add_colorbar=False)
axes[0].set_title('Hillshade (azimuth=315)')
axes[0].set_axis_off()

svf_final.plot.imshow(ax=axes[1], cmap='gray', vmin=0, vmax=1, add_colorbar=True,
                      cbar_kwargs={'label': 'SVF'})
axes[1].set_title('Sky-View Factor')
axes[1].set_axis_off()

plt.tight_layout()

# Save preview image
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/sky_view_factor_preview.png', bbox_inches='tight', dpi=120)

<div class="alert alert-block alert-warning">
<b>Resolution and radius units.</b> The <code>max_radius</code> parameter is in cells, not meters. On a 1m DEM, <code>max_radius=30</code> searches 30 meters. On a 30m DEM, the same value searches 900 meters. Scale the radius to match your feature size and DEM resolution.
</div>

### References

- [Sky view factor (Wikipedia)](https://en.wikipedia.org/wiki/Sky_view_factor)
- Zaksek, K., Ostir, K., & Kokalj, Z. (2011). [Sky-View Factor as a Relief Visualization Technique](https://doi.org/10.3390/rs3020398). *Remote Sensing*, 3(2), 398-415.
- [xrspatial.sky_view_factor API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.sky_view_factor.html)